# Experimental Visualization of SVC

## Set Up

Libraries/packages

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.data_utils import get_data, get_models
from src.config import BASE_PATH
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyvista as pv

Data/Models

In [ ]:
# Data
DATA_DICT = get_data(is_nomo=False)
HOPKINS_DICT = {
    "X": pd.read_parquet(
        BASE_PATH / "data" / "processed" / "hopkins" / "base" / "X.parquet"
    ),
    "y": pd.read_excel(
        BASE_PATH / "data" / "processed" / "hopkins" / "base" / "y.xlsx",
        index_col=0,
    ),
}

# Models
model_dir = BASE_PATH / "models" / "trained"
## Base models
svc_model = get_models(["svc"], model_dir)["svc"]

## 3D

In [ ]:
pv.OFF_SCREEN = True
X = HOPKINS_DICT["X"]
y = HOPKINS_DICT["y"].values.ravel()

# Reduce to 3D with PCA
pca = PCA(n_components=3)
X_reduced = pca.fit_transform(X)

# Create 3D meshgrid for decision boundary
x_min, x_max = X_reduced[:, 0].min() - 1, X_reduced[:, 0].max() + 1
y_min, y_max = X_reduced[:, 1].min() - 1, X_reduced[:, 1].max() + 1
z_min, z_max = X_reduced[:, 2].min() - 1, X_reduced[:, 2].max() + 1

xx, yy, zz = np.meshgrid(
    np.linspace(x_min, x_max, 100),
    np.linspace(y_min, y_max, 100),
    np.linspace(z_min, z_max, 100),
)

# Transform grid points back to 50D, then predict with original model
grid_3d = np.c_[xx.ravel(), yy.ravel(), zz.ravel()]
grid_50d = pca.inverse_transform(grid_3d)
Z = svc_model.predict(grid_50d)

# Reshape Z to match meshgrid shape
Z = Z.reshape(xx.shape)

# Find points where prediction changes (near decision boundary)
decision_values = svc_model.decision_function(grid_50d)
boundary_mask = np.abs(decision_values) < 0.1

boundary_points = grid_3d[boundary_mask]

# PyVista visualization
plotter = pv.Plotter(window_size=[1200, 900])

# Add class 0 points
class_0_points = X_reduced[y == 0]
if len(class_0_points) > 0:
    cloud_0 = pv.PolyData(class_0_points)
    plotter.add_mesh(
        cloud_0,
        color="blue",
        point_size=12,
        render_points_as_spheres=True,
        label="Class 0",
        opacity=0.8,
    )

# Add class 1 points
class_1_points = X_reduced[y == 1]
if len(class_1_points) > 0:
    cloud_1 = pv.PolyData(class_1_points)
    plotter.add_mesh(
        cloud_1,
        color="red",
        point_size=12,
        render_points_as_spheres=True,
        label="Class 1",
        opacity=0.8,
    )

# Add decision boundary
if len(boundary_points) > 0:
    boundary_cloud = pv.PolyData(boundary_points)
    plotter.add_mesh(
        boundary_cloud,
        color="green",
        point_size=6,
        render_points_as_spheres=True,
        label="Decision Boundary",
        opacity=0.4,
    )

# Add axes and labels
plotter.add_axes(xlabel="PC1", ylabel="PC2", zlabel="PC3")
plotter.add_legend(bcolor="white", face="rectangle", size=[0.15, 0.15])

# Set camera view (similar to matplotlib view_init)
plotter.camera_position = "iso"

# Add title
plotter.add_text(
    "SVC Decision Boundary (Poly Kernel, degree=3)",
    position="upper_edge",
    font_size=14,
    color="black",
)

# Show the plot
plotter.show()

# Optional: Save high-resolution image
# plotter.screenshot('svc_boundary_3d.png', window_size=[2400, 1800])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

X = HOPKINS_DICT["X"]
y = HOPKINS_DICT["y"].values.ravel()

# Reduce to 2D with PCA (changed from 3)
pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X)

# Create 2D meshgrid for decision boundary
x_min, x_max = X_reduced[:, 0].min() - 1, X_reduced[:, 0].max() + 1
y_min, y_max = X_reduced[:, 1].min() - 1, X_reduced[:, 1].max() + 1

xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))

# Transform grid points back to 50D, then predict with original model
grid_2d = np.c_[xx.ravel(), yy.ravel()]
grid_50d = pca.inverse_transform(grid_2d)
Z = svc_model.predict(grid_50d)

# Reshape Z to match meshgrid shape
Z = Z.reshape(xx.shape)

# Create the plot
fig, ax = plt.subplots(figsize=(12, 10))

# Plot decision boundary as contour
contour = ax.contourf(xx, yy, Z, levels=1, alpha=0.3, colors=["blue", "red"])

# Plot decision boundary line
ax.contour(xx, yy, Z, levels=[0.5], linewidths=3, colors="green")

# Plot the data points
scatter0 = ax.scatter(
    X_reduced[y == 0, 0],
    X_reduced[y == 0, 1],
    c="blue",
    s=80,
    edgecolors="black",
    linewidth=1,
    label="Class 0",
    alpha=0.8,
)
scatter1 = ax.scatter(
    X_reduced[y == 1, 0],
    X_reduced[y == 1, 1],
    c="red",
    s=80,
    edgecolors="black",
    linewidth=1,
    label="Class 1",
    alpha=0.8,
)

# Labels and styling
ax.set_xlabel("PC1", fontsize=14, fontweight="bold")
ax.set_ylabel("PC2", fontsize=14, fontweight="bold")
ax.set_title(
    "SVC Decision Boundary (Poly Kernel, degree=3)",
    fontsize=16,
    fontweight="bold",
    pad=20,
)
ax.legend(fontsize=12, loc="best")
ax.grid(True, alpha=0.3, linestyle="--")

plt.tight_layout()
# plt.savefig('svc_boundary_2d.png', dpi=300, bbox_inches='tight')
plt.show()